# MLflow experiment: Decision Tree loan default prediction

**Goal:** Predict `loan_default` from the banking CSV and record five training attempts as **five runs inside this notebook's own MLflow experiment**.

- `0` means no default; `1` means default. We treat `1` as the positive class.
- Run the cells **in order**. Each numbered run is a separate notebook cell, so its MLflow record appears immediately.
- The decision tree and random forest notebooks use **different experiment names**. They share the same split (`random_state=42`) for a fair comparison.
- These are learning examples. Never choose a model solely by accuracy when the cost of missed defaults matters.


## 1. Setup

Install the packages in the notebook's Python environment. Uncomment the installation line if needed. Start `mlflow server` in a **separate terminal**, then open http://127.0.0.1:5000. Leave the terminal running while executing this notebook. `set_tracking_uri` sends runs to that server. A local server keeps experiment metadata in a local SQLite database by default.

In [6]:
# Uncomment if packages are missing, then restart the kernel.
# %pip install mlflow pandas scikit-learn matplotlib

import mlflow
import mlflow.sklearn
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, ConfusionMatrixDisplay)


## 2. Read and inspect the banking data

Place `Banking_Loan_Default_Classification(2).csv` in the **same folder** as this notebook. `loan_default` is the target column. Check row count, missing values, and class balance before modeling.

In [7]:
df = pd.read_csv("Banking_Loan_Default_Classification.csv")
print("Shape:", df.shape)
print("Missing cells:", df.isna().sum().sum())
print("Target counts:\n", df["loan_default"].value_counts())
display(df.head())
assert "loan_default" in df.columns
assert set(df["loan_default"].dropna().unique()) == {0, 1}


Shape: (1000, 20)
Missing cells: 0
Target counts:
 loan_default
0    613
1    387
Name: count, dtype: int64


,age,monthly_income_inr,employment_type,years_employed,education_level,marital_status,dependents,residence_type,years_at_address,loan_amount_inr,loan_term_months,loan_purpose,existing_loans,monthly_debt_payments_inr,credit_score,late_payments_last_12m,savings_balance_inr,account_tenure_years,has_guarantor,loan_default
0,32,56000,Self-employed,4,Graduate,Single,2,Family,2,990000,12,Medical,0,14000,678,0,265000,3,No,0
1,39,42000,Self-employed,11,Graduate,Single,0,Rented,21,100000,36,Education,2,17000,724,0,170000,3,No,0
2,60,60000,Self-employed,17,High school,Single,1,Family,16,240000,24,Home improvement,1,23000,641,1,225000,14,No,0
3,30,67000,Self-employed,0,Graduate,Single,0,Owned,12,890000,60,Vehicle,1,21000,597,1,95000,12,No,0
4,30,50000,Salaried,4,Postgraduate,Single,2,Owned,12,150000,60,Medical,1,11000,633,3,245000,12,Yes,1


## 3. Make reproducible training, validation, and test sets

Use **60% training**, **20% validation**, and **20% test**. `stratify` keeps roughly the same default proportion in each split. Fit the preprocessing steps only on the training portion. Compare all five runs on the **same validation set**; reserve the test set for the selected model at the end.
Numeric input columns are converted to `float64` before splitting. This allows MLflow to infer a schema that can accept missing numeric values at prediction time; the pipeline imputes those values.


In [ ]:
X = df.drop(columns="loan_default").copy()
y = df["loan_default"]

# Float columns accept missing values in MLflow's inferred input schema.
# The median imputer in the pipeline still handles missing values.
numeric_input_columns = X.select_dtypes(include=["number"]).columns
X[numeric_input_columns] = X[numeric_input_columns].astype("float64")
X_train, X_holdout, y_train, y_holdout = train_test_split(
    X, y, test_size=0.40, random_state=42, stratify=y
)
X_valid, X_test, y_valid, y_test = train_test_split(
    X_holdout, y_holdout, test_size=0.50, random_state=42,
    stratify=y_holdout
)
print("Train:", X_train.shape, "Validation:", X_valid.shape, "Test:", X_test.shape)
print("Default rates:", round(y_train.mean(), 3),
      round(y_valid.mean(), 3), round(y_test.mean(), 3))

categorical_columns = X.select_dtypes(include=["object", "category"]).columns.tolist()
numeric_columns = X.select_dtypes(exclude=["object", "category"]).columns.tolist()
preprocess = ColumnTransformer([
    ("numeric", SimpleImputer(strategy="median"), numeric_columns),
    ("categorical", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("one_hot", OneHotEncoder(handle_unknown="ignore"))
    ]), categorical_columns)
])
print("Numeric:", numeric_columns)
print("Categorical:", categorical_columns)


## 4. Connect to the Decision Tree experiment

An **experiment** groups the attempts. A **run** records one fitted model and its settings (**parameters**), validation scores (**metrics**), and saved pipeline (**model artifact**). Running a numbered cell again creates another run, even if its settings are unchanged.

`mlflow server` must already be running. This experiment name belongs only to this notebook.

In [9]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("Banking_Loan_Default_Decision_Tree")
print("Connected to:", mlflow.get_tracking_uri())


2026/09/25 13:17:07 INFO mlflow.tracking.fluent: Experiment with name 'Banking_Loan_Default_Decision_Tree' does not exist. Creating a new experiment.


Connected to: http://127.0.0.1:5000


## 5. One training function, called once per run

The helper trains a **new** `DecisionTreeClassifier` for every call. It logs the chosen `max_depth` and `min_samples_leaf`, and evaluates the positive class `1` on validation data. `max_depth` limits the number of splits along a path. `min_samples_leaf` requires a minimum number of training records at each leaf. Larger values usually make the tree simpler; **better validation performance is not guaranteed**.

Accuracy is the overall proportion correct. Precision asks “of cases predicted default, how many did default?” Recall asks “of actual defaults, how many did we find?” F1 balances precision and recall. ROC AUC measures ranking across thresholds. We save the preprocessing pipeline with the model so the same transformations can be applied later.
The current MLflow `skops` format checks model types before saving. This scikit-learn tree pipeline uses `numpy.dtype` and `sklearn.tree._tree.Tree`, so the logging call explicitly trusts those two types for this locally trained model. Only trust these types for models whose source you know.


In [ ]:
from sklearn.tree import DecisionTreeClassifier

def train_and_log(run_name, max_depth, min_samples_leaf):
    with mlflow.start_run(run_name=run_name) as run:
        model = DecisionTreeClassifier(
            max_depth=max_depth,
            min_samples_leaf=min_samples_leaf,
            random_state=42
        )
        pipeline = Pipeline([("preprocess", preprocess), ("model", model)])
        pipeline.fit(X_train, y_train)
        predicted = pipeline.predict(X_valid)
        probabilities = pipeline.predict_proba(X_valid)[:, 1]
        scores = {
            "valid_accuracy": accuracy_score(y_valid, predicted),
            "valid_precision": precision_score(y_valid, predicted, zero_division=0),
            "valid_recall": recall_score(y_valid, predicted, zero_division=0),
            "valid_f1": f1_score(y_valid, predicted, zero_division=0),
            "valid_roc_auc": roc_auc_score(y_valid, probabilities)
        }
        mlflow.log_params({"algorithm": "DecisionTreeClassifier",
                           "max_depth": str(max_depth),
                           "min_samples_leaf": min_samples_leaf,
                           "random_state": 42})
        mlflow.log_metrics(scores)
        mlflow.set_tags({"dataset": "Banking_Loan_Default_Classification.csv",
                         "positive_class": "1", "split": "60_train_20_valid_20_test"})
        mlflow.sklearn.log_model(sk_model=pipeline, name="loan_default_pipeline",
                                 input_example=X_train.head(3),
                                 skops_trusted_types=[
                                     "numpy.dtype",
                                     "sklearn.tree._tree.Tree"
                                 ])
        print("Run:", run_name, "ID:", run.info.run_id)
        print(pd.Series(scores).round(3).to_string())
        return run.info.run_id


### Run 1 - shallow baseline

Start with a shallow tree. A small depth is easier to interpret but may miss important patterns. Each run logs the same set of validation metrics. Compare `valid_recall` and `valid_f1`; an increase is possible but not promised.

In [11]:
run_1_id = train_and_log("depth_2_leaf_1", max_depth=2, min_samples_leaf=1)


c:\Users\admin\Desktop\programs\venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
c:\Users\admin\Desktop\programs\venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains m

🏃 View run depth_2_leaf_1 at: http://127.0.0.1:5000/#/experiments/1/runs/68cf1aa376cc4ed1a8b4ead6388a0621
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


MlflowException: The saved sklearn model references untrusted types. If you are sure loading these types is safe, set the 'skops_trusted_types' parameter when calling 'log_model' or 'save_model' to the list of trusted types. Root error: Untrusted types found in the file: ['numpy.dtype', 'sklearn.tree._tree.Tree'].

- sklearn.tree._tree.Tree: sklearn.tree._tree.Tree (the shared node storage for DecisionTree*, RandomForest*, ExtraTrees*, and GradientBoosting* models) stores raw node indices (left_child, right_child, feature) that scikit-learn indexes into without bounds checking. A malicious file can set these to out-of-range values: skops loads the object successfully, but calling .predict() on it can then crash the process (segfault) or read out-of-bounds memory. If you created the file yourself or otherwise fully trust its source, you can load it with trusted=["sklearn.tree._tree.Tree"].

Only add the specific types you have reviewed and trust to the `trusted` argument; avoid passing 

### Run 2 - increase depth

Only `max_depth` changes from 2 to 4. Did recall improve? Each run logs the same set of validation metrics. Compare `valid_recall` and `valid_f1`; an increase is possible but not promised.

In [ ]:
run_2_id = train_and_log("depth_4_leaf_1", max_depth=4, min_samples_leaf=1)


### Run 3 - increase depth again

Only `max_depth` changes from 4 to 6. Check whether a deeper tree helps validation performance. Each run logs the same set of validation metrics. Compare `valid_recall` and `valid_f1`; an increase is possible but not promised.

In [ ]:
run_3_id = train_and_log("depth_6_leaf_1", max_depth=6, min_samples_leaf=1)


### Run 4 - regularize leaves

Hold depth at 6 and change `min_samples_leaf` from 1 to 5. This discourages tiny leaves. Each run logs the same set of validation metrics. Compare `valid_recall` and `valid_f1`; an increase is possible but not promised.

In [ ]:
run_4_id = train_and_log("depth_6_leaf_5", max_depth=6, min_samples_leaf=5)


### Run 5 - regularize further

Hold depth at 6 and increase `min_samples_leaf` from 5 to 10. Does validation F1 rise or fall? Each run logs the same set of validation metrics. Compare `valid_recall` and `valid_f1`; an increase is possible but not promised.

In [ ]:
run_5_id = train_and_log("depth_6_leaf_10", max_depth=6, min_samples_leaf=10)


## 6. Compare the five runs

`search_runs` reads the runs from **this experiment only**. Sort by validation F1, then inspect recall and precision before deciding. Run IDs let you retrieve the exact model associated with each score. If you execute a run cell twice, the table will include the extra run; use the first five freshly created IDs listed below to compare the intended sequence.

In [ ]:
experiment = mlflow.get_experiment_by_name("Banking_Loan_Default_Decision_Tree")
run_ids = [run_1_id, run_2_id, run_3_id, run_4_id, run_5_id]
runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id])
runs = runs[runs["run_id"].isin(run_ids)].copy()
columns = ["tags.mlflow.runName", "run_id", "params.max_depth",
           "params.min_samples_leaf", "metrics.valid_accuracy",
           "metrics.valid_precision", "metrics.valid_recall",
           "metrics.valid_f1", "metrics.valid_roc_auc"]
comparison = runs[columns].sort_values("metrics.valid_f1", ascending=False)
display(comparison.round(3))
print("Open http://127.0.0.1:5000 to inspect the same experiment in the UI.")


## 7. Select once, then evaluate on the held-out test set

Select the run with highest validation F1 **for this exercise**. A real bank may choose a different criterion after considering the cost of missed defaults and false alerts. Load the exact saved pipeline from the winning run and calculate test metrics only now. The test result is a final estimate, not a reason to retune the same test set repeatedly.

In [ ]:
best_run_id = comparison.iloc[0]["run_id"]
model_uri = f"runs:/{best_run_id}/loan_default_pipeline"
best_pipeline = mlflow.sklearn.load_model(model_uri)
test_predictions = best_pipeline.predict(X_test)
test_probabilities = best_pipeline.predict_proba(X_test)[:, 1]
test_scores = {
    "test_accuracy": accuracy_score(y_test, test_predictions),
    "test_precision": precision_score(y_test, test_predictions, zero_division=0),
    "test_recall": recall_score(y_test, test_predictions, zero_division=0),
    "test_f1": f1_score(y_test, test_predictions, zero_division=0),
    "test_roc_auc": roc_auc_score(y_test, test_probabilities)
}
print("Selected run:", best_run_id)
print(pd.Series(test_scores).round(3).to_string())
ConfusionMatrixDisplay.from_predictions(
    y_test, test_predictions, display_labels=["No default", "Default"],
    cmap="Blues", values_format="d"
)
plt.title("Held-out test confusion matrix")
plt.show()


## 8. What to conclude

1. Which specific hyperparameter change improved validation F1? Which did not?
2. Did recall and precision move in the same direction? Explain the tradeoff for loan default screening.
3. Does the selected model's test F1 look close to its validation F1? A gap may indicate selection noise or overfitting.
4. In the MLflow UI, open the best run. Identify its parameters, metrics, tags, and saved model.

**Terms:** experiment = related runs; run = one training attempt; parameter = chosen setting; metric = measured score; artifact = saved model or file. The two notebooks intentionally use different experiments, with five runs each. Reexecuting cells creates additional runs.